# Endpoint-Pairwise Decoding Eval (100 samples)

기존 추론 결과 JSON/JSONL에 저장된 `first_probs`, `last_probs`, `pair_probs`를 재사용해서 모델 재추론 없이 decoder만 비교합니다.

비교 대상:

- `current_weighted_permutation`: 기존 결과의 `CURRENT_DECODER_FIELD` 또는 weighted fallback
- `endpoint_pair_50_50`: endpoint classifier 0.5 + pairwise-derived endpoint 0.5
- `endpoint_pair_40_60`: endpoint classifier 0.4 + pairwise-derived endpoint 0.6
- `direct_order`: 결과 파일에 존재하면 참고 baseline으로 함께 평가

기본 설정은 로드한 결과 중 앞 100개만 사용합니다. `RESULT_PATH`는 직접 지정할 수 있습니다. 캐시가 없으면 지정된 adapter로 100개 샘플의 probability cache를 먼저 생성한 뒤 평가합니다.


In [ ]:
# 1) Config
import itertools
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd

# 기존 추론 결과 JSON/JSONL 경로를 직접 지정하세요.
# 이 파일에는 sample별 gold_order, first_probs, last_probs, pair_probs가 있어야 합니다.
# 예:
# RESULT_PATH = (
#     '/content/drive/MyDrive/SNU_AI_Challenge/'
#     'qwen2vl_lgt_order_refine_v1/runs/20260714_003635/'
#     'lgt_order_refine/eval/<정확한>_task_probability_cache.json'
# )
RESULT_PATH = None

# 캐시가 없을 때 이 노트북에서 100개 샘플 확률 cache를 직접 만들지 여부입니다.
# True로 두면 adapter를 로드해서 first/last/pairwise/direct 추론을 수행합니다.
CREATE_CACHE_IF_MISSING = True
CACHE_ADAPTER_DIR = (
    '/content/drive/MyDrive/SNU_AI_Challenge/'
    'qwen2vl_lgt_order_refine_v1/runs/20260714_003635/'
    'lgt_order_refine/best_adapter'
)
CACHE_OUTPUT_PATH = (
    '/content/drive/MyDrive/SNU_AI_Challenge/'
    'qwen2vl_lgt_order_refine_v1/runs/20260714_003635/'
    'lgt_order_refine/eval/best_adapter_eval100_task_probability_cache.json'
)


# RESULT_PATH가 None이면 아래 패턴으로 후보만 출력합니다.
CANDIDATE_RESULT_GLOBS = [
    '/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/eval/*task_probability_cache*.json',
    '/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/eval/*task_probability_cache*.jsonl',
    '/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/**/eval/*task_probability_cache*.json',
    '/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/**/eval/*task_probability_cache*.jsonl',
]

SAMPLE_LIMIT = 100
CURRENT_DECODER_FIELD = 'weighted_order'

# CURRENT_DECODER_FIELD가 없을 때만 사용하는 fallback 가중치입니다.
# 기존 weighted permutation decoder가 다른 값을 썼다면 여기만 바꾸면 됩니다.
CURRENT_WEIGHT_ALPHA = 1.0  # pairwise
CURRENT_WEIGHT_BETA = 1.0   # first
CURRENT_WEIGHT_GAMMA = 1.0  # last

OUTPUT_DIR = Path('/content/drive/MyDrive/SNU_AI_Challenge/results/eval_endpoint_pairwise_decoding')
if not Path('/content').exists():
    OUTPUT_DIR = Path('results/eval_endpoint_pairwise_decoding')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# In Colab, do not create /content/drive/... here. Create it after Drive is mounted.

FRAME_IDS = [1, 2, 3, 4]
PERMUTATIONS = list(itertools.permutations(FRAME_IDS))
EPS = 1e-12

print('output_dir:', OUTPUT_DIR)


In [ ]:
# 2) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path('/content/.snu_lgt_order_refine_deps_installed')

if Path('/content').exists() and not MARKER.exists():
    packages = [
        'transformers>=4.49.0,<4.54.0',
        'accelerate>=0.34.0',
        'bitsandbytes>=0.46.1',
        'peft',
        'qwen-vl-utils',
        'modelscope',
        'jedi',
        'pandas==2.2.2',
        'safetensors>=0.4.5',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])
    MARKER.write_text('ok')
    print('Dependencies installed. Restarting runtime. Run this cell again after restart.')
    os.kill(os.getpid(), 9)
elif Path('/content').exists():
    print('Dependencies already installed. Continue.')
else:
    print('Local environment detected. Skipping Colab dependency install.')


In [ ]:
# 3) Optional: create probability cache for this adapter when RESULT_PATH is missing
# This cell mounts Drive and runs model inference only when CREATE_CACHE_IF_MISSING=True and RESULT_PATH is None/missing.
def maybe_create_probability_cache():
    global RESULT_PATH
    if RESULT_PATH is not None and Path(RESULT_PATH).exists():
        print('Using existing RESULT_PATH:', RESULT_PATH)
        return
    if not CREATE_CACHE_IF_MISSING:
        print('Cache creation disabled. Set RESULT_PATH manually or set CREATE_CACHE_IF_MISSING=True.')
        return

    if not Path('/content').exists():
        raise RuntimeError('Cache creation is intended for Colab because it needs Drive data/model paths.')

    import ast
    import gc
    import random
    import re
    import shutil
    import zipfile

    from google.colab import drive
    drive_root = Path('/content/drive')
    expected_snu_root = Path('/content/drive/MyDrive/SNU_AI_Challenge')

    # If Config created local folders under /content/drive before mount, Colab refuses to mount.
    # Remove only the unmounted local mountpoint, then recreate it.
    if drive_root.exists() and not os.path.ismount(str(drive_root)) and any(drive_root.iterdir()):
        print('Removing local pre-mount /content/drive contents:', sorted(str(p) for p in drive_root.iterdir())[:20])
        shutil.rmtree(drive_root)
    drive_root.mkdir(parents=True, exist_ok=True)

    print('Mounting Google Drive...')
    drive.mount('/content/drive')

    if not expected_snu_root.exists():
        mydrive_root = Path('/content/drive/MyDrive')
        visible = sorted(str(p) for p in mydrive_root.iterdir())[:50] if mydrive_root.exists() else []
        print('Visible MyDrive entries:', visible)
        raise FileNotFoundError(f'Drive mounted, but expected folder is missing: {expected_snu_root}')
    print('Drive mounted with SNU root:', expected_snu_root)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


    import torch
    from PIL import Image
    from tqdm.auto import tqdm
    from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, set_seed
    try:
        from transformers import Qwen2VLForConditionalGeneration
    except ImportError:
        Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
    from peft import PeftModel
    from transformers.utils import logging as transformers_logging

    transformers_logging.set_verbosity_error()
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

    model_repo_id = 'Qwen/Qwen2-VL-2B-Instruct'

    def resolve_snu_root():
        candidates = [Path('/content/drive/MyDrive/SNU_AI_Challenge')]
        candidates.extend(Path('/content/drive/MyDrive').glob('*AI*Challenge*'))
        for root in candidates:
            if (root / 'snuaichallenge.zip').exists() or (root / 'model_cache/Qwen2-VL-2B-Instruct').exists() or (root / 'id_splits').exists():
                print('Using SNU root:', root)
                return root
        raise FileNotFoundError('Could not find SNU_AI_Challenge root under /content/drive/MyDrive')

    snu_root = resolve_snu_root()
    drive_model_dir = str(snu_root / 'model_cache/Qwen2-VL-2B-Instruct')

    def resolve_split_dir():
        id_splits_root = snu_root / 'id_splits'
        preferred = [
            id_splits_root / 'qwen2vl_lgt_order_refine_20260714_003635',
            id_splits_root / 'qwen2vl_lgt_order_refien_20260714_003635',
        ]
        candidates = preferred + sorted(id_splits_root.glob('qwen2vl_lgt_order_refin*20260714_003635*'))
        for path in candidates:
            if (path / 'tuning150_ids.json').exists():
                print('Using split dir:', path)
                return str(path)
        found = sorted(id_splits_root.glob('**/tuning150_ids.json')) if id_splits_root.exists() else []
        if found:
            print('Using discovered split dir:', found[0].parent)
            return str(found[0].parent)
        raise FileNotFoundError(f'Could not find tuning150_ids.json under {id_splits_root}')

    split_dir = resolve_split_dir()

    def resolve_data_paths():
        data_candidates = [
            Path('/content/snuaichallenge_data'),
            Path('/content/snuaichallenge'),
            Path('/content/data'),
            snu_root / 'snuaichallenge_data',
            snu_root / 'snuaichallenge',
            snu_root / 'data',
        ]
        for root in data_candidates:
            train_csv_path = root / 'train.csv'
            train_image_path = root / 'train'
            if train_csv_path.exists() and train_image_path.is_dir():
                print('Using data dir:', root)
                return str(root), str(train_csv_path), str(train_image_path)

        search_roots = [snu_root, Path('/content')]
        found_train_csvs = []
        for root in search_roots:
            if root.exists():
                found_train_csvs.extend(root.glob('**/train.csv'))
        for train_csv_path in sorted(found_train_csvs, key=lambda x: str(x)):
            root = train_csv_path.parent
            train_image_path = root / 'train'
            if train_image_path.is_dir():
                print('Using discovered data dir:', root)
                return str(root), str(train_csv_path), str(train_image_path)

        zip_candidates = [
            snu_root / 'snuaichallenge.zip',
            snu_root / 'data/snuaichallenge.zip',
            snu_root / 'dataset/snuaichallenge.zip',
            Path('/content/snuaichallenge.zip'),
        ]
        zip_candidates.extend(snu_root.glob('**/*snuaichallenge*.zip'))
        zip_candidates = [path for path in dict.fromkeys(zip_candidates) if path.exists()]
        if zip_candidates:
            zip_path = zip_candidates[0]
            print('Extracting data zip:', zip_path)
            with zipfile.ZipFile(zip_path) as zip_file:
                zip_file.extractall('/content/')
            for root in data_candidates:
                train_csv_path = root / 'train.csv'
                train_image_path = root / 'train'
                if train_csv_path.exists() and train_image_path.is_dir():
                    print('Using extracted data dir:', root)
                    return str(root), str(train_csv_path), str(train_image_path)

        checked = [str(path) for path in data_candidates]
        raise FileNotFoundError(
            'Could not find dataset. Expected a directory containing train.csv and train/. '
            f'Checked: {checked}. Also searched under /content and the detected SNU root.'
        )

    data_dir, train_csv, train_image_dir = resolve_data_paths()

    def ensure_base_model_path():
        if os.path.exists(os.path.join(drive_model_dir, 'config.json')):
            return drive_model_dir
        from modelscope import snapshot_download as modelscope_snapshot_download
        model_dir = modelscope_snapshot_download(model_repo_id, cache_dir='/content/modelscope_cache')
        os.makedirs(os.path.dirname(drive_model_dir), exist_ok=True)
        if not os.path.exists(drive_model_dir):
            shutil.copytree(model_dir, drive_model_dir)
        return drive_model_dir

    for required in [train_csv, train_image_dir, CACHE_ADAPTER_DIR]:
        assert os.path.exists(required), required

    seed = 42
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    def parse_answer(answer):
        result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
        result = [int(value) for value in result]
        if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
            raise ValueError(f'Invalid Answer: {answer}')
        return result

    def order_to_sequence(answer):
        return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]

    def row_image_paths(row):
        sample_id = str(row['Id'])
        return [os.path.join(train_image_dir, sample_id, str(row[f'Input_{i}'])) for i in range(1, 5)]

    def load_rgb(path):
        with Image.open(path) as image:
            return image.convert('RGB').copy()

    def rows_by_ids(dataframe, ids):
        ids = [str(value) for value in ids]
        subset = dataframe[dataframe['Id'].isin(ids)].copy()
        order = {sample_id: index for index, sample_id in enumerate(ids)}
        subset['_split_order'] = subset['Id'].map(order)
        return subset.sort_values('_split_order').drop(columns=['_split_order']).reset_index(drop=True)

    train_df = pd.read_csv(train_csv)
    train_df['Id'] = train_df['Id'].astype(str)
    train_df['Answer_list'] = train_df['Answer'].apply(parse_answer)
    if os.path.exists(os.path.join(split_dir, 'tuning150_ids.json')):
        with open(os.path.join(split_dir, 'tuning150_ids.json'), 'r', encoding='utf-8') as f:
            eval_rows = rows_by_ids(train_df, json.load(f)).head(SAMPLE_LIMIT).reset_index(drop=True)
    else:
        eval_rows = train_df.sample(n=min(SAMPLE_LIMIT, len(train_df)), random_state=seed).reset_index(drop=True)

    def task_instruction(example):
        sentence = example['sentence']
        task_type = example['task_type']
        if task_type == 'pairwise':
            return (
                f'Caption:\n{sentence}\n\n'
                'Question: Which image occurs first?\n'
                'If the first image occurs earlier, answer 1.\n'
                'If the second image occurs earlier, answer 2.\n'
                'Answer only 1 or 2.'
            )
        if task_type == 'first':
            return f'Caption:\n{sentence}\n\nQuestion: Which image represents the beginning of the story?\nAnswer only the image number from 1 to 4.'
        if task_type == 'last':
            return f'Caption:\n{sentence}\n\nQuestion: Which image represents the end of the story?\nAnswer only the image number from 1 to 4.'
        if task_type == 'order':
            return (
                f'Caption:\n{sentence}\n\n'
                'Question: Compare the temporal relation between scenes and identify the likely first and last scenes.\n'
                'Using these cues, determine the complete chronological order.\n'
                'Output only the final ordered list, such as [1, 2, 3, 4].'
            )
        raise ValueError(task_type)

    def make_messages(example, include_answer=False):
        content = []
        for idx, _ in enumerate(example['image_paths'], start=1):
            content.append({'type': 'text', 'text': f'\nImage {idx}:'})
            content.append({'type': 'image'})
        content.append({'type': 'text', 'text': '\n\n' + task_instruction(example)})
        messages = [{'role': 'user', 'content': content}]
        if include_answer:
            messages.append({'role': 'assistant', 'content': str(example['target'])})
        return messages

    def make_eval_example(row, task_type, pair=None):
        answer = [int(value) for value in row.get('Answer_list', [1, 2, 3, 4])]
        image_paths = row_image_paths(row)
        example = {
            'sample_id': str(row['Id']),
            'sentence': '' if pd.isna(row['Sentence']) else str(row['Sentence']),
            'answer': answer,
            'order': order_to_sequence(answer),
            'image_paths': image_paths,
            'task_type': task_type,
            'target': '1',
        }
        if task_type == 'pairwise':
            a, b = pair
            example['image_paths'] = [image_paths[a - 1], image_paths[b - 1]]
        return example

    def parse_order_prediction(text):
        match = re.fullmatch(r'\s*\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]\s*', str(text))
        if not match:
            return None
        values = [int(value) for value in match.groups()]
        return values if sorted(values) == [1, 2, 3, 4] else None

    model_id = ensure_base_model_path()
    model_local_files_only = os.path.isdir(model_id)
    processor = AutoProcessor.from_pretrained(
        model_id,
        min_pixels=128 * 28 * 28,
        max_pixels=256 * 28 * 28,
        local_files_only=model_local_files_only,
    )
    processor.tokenizer.padding_side = 'right'
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base = Qwen2VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map='auto',
        local_files_only=model_local_files_only,
    )
    model = PeftModel.from_pretrained(base, CACHE_ADAPTER_DIR, is_trainable=False)
    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    model.eval()

    def model_device():
        return next(model.parameters()).device

    def digit_token_id(digit):
        ids = processor.tokenizer.encode(str(digit), add_special_tokens=False)
        if len(ids) != 1:
            raise ValueError(f'Digit {digit} tokenized to {ids}')
        return ids[0]

    digit_token_ids = {digit: digit_token_id(digit) for digit in [1, 2, 3, 4]}

    @torch.no_grad()
    def score_digit_candidates(example, candidates):
        old_padding_side = processor.tokenizer.padding_side
        processor.tokenizer.padding_side = 'right'
        text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
        images = [load_rgb(path) for path in example['image_paths']]
        inputs = processor(text=[text], images=[images], return_tensors='pt')
        inputs = {key: value.to(model_device()) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = model(**inputs)
        last_pos = int(inputs['attention_mask'][0].sum().item()) - 1
        logits = outputs.logits[0, last_pos]
        token_ids = [digit_token_ids[int(candidate)] for candidate in candidates]
        probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
        processor.tokenizer.padding_side = old_padding_side
        return {int(candidate): float(prob) for candidate, prob in zip(candidates, probs)}

    @torch.no_grad()
    def generate_order(row, max_new_tokens=16):
        example = make_eval_example(row, 'order')
        text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
        images = [load_rgb(path) for path in example['image_paths']]
        old_padding_side = processor.tokenizer.padding_side
        processor.tokenizer.padding_side = 'left'
        inputs = processor(text=[text], images=[images], return_tensors='pt')
        inputs = {key: value.to(model_device()) if torch.is_tensor(value) else value for key, value in inputs.items()}
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
        new_tokens = generated[:, inputs['input_ids'].shape[1]:]
        output = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
        processor.tokenizer.padding_side = old_padding_side
        return parse_order_prediction(output), output

    pair_indices = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
    records = []
    for _, row in tqdm(eval_rows.iterrows(), total=len(eval_rows), desc='best_adapter eval100 probs'):
        answer = [int(value) for value in row['Answer_list']]
        gold_order = order_to_sequence(answer)
        first_probs = score_digit_candidates(make_eval_example(row, 'first'), [1, 2, 3, 4])
        last_probs = score_digit_candidates(make_eval_example(row, 'last'), [1, 2, 3, 4])
        pair_probs = {}
        pair_correct = []
        for first_index, second_index in pair_indices:
            a, b = first_index + 1, second_index + 1
            probs = score_digit_candidates(make_eval_example(row, 'pairwise', pair=(a, b)), [1, 2])
            p_a_before_b = probs[1]
            pair_probs[f'{a}>{b}'] = float(p_a_before_b)
            pair_probs[f'{b}>{a}'] = float(1.0 - p_a_before_b)
            pred_first = a if p_a_before_b >= 0.5 else b
            gold_first = a if answer[first_index] < answer[second_index] else b
            pair_correct.append(int(pred_first == gold_first))
        direct_order, direct_text = generate_order(row)
        records.append({
            'sample_id': str(row['Id']),
            'gold_order': gold_order,
            'first_probs': {str(k): v for k, v in first_probs.items()},
            'last_probs': {str(k): v for k, v in last_probs.items()},
            'pair_probs': pair_probs,
            'direct_order': direct_order,
            'direct_text': direct_text,
            'pairwise_accuracy': float(np.mean(pair_correct)),
            'first_accuracy': float(max(first_probs, key=first_probs.get) == gold_order[0]),
            'last_accuracy': float(max(last_probs, key=last_probs.get) == gold_order[-1]),
        })

    cache_path = Path(CACHE_OUTPUT_PATH)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open('w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    RESULT_PATH = str(cache_path)
    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    print('created cache:', RESULT_PATH)


maybe_create_probability_cache()


In [ ]:
# 4) Load existing inference results
def load_json_or_jsonl(path):
    path = Path(path)
    if path.suffix.lower() == '.jsonl':
        with path.open('r', encoding='utf-8') as f:
            return [json.loads(line) for line in f if line.strip()]
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    for key in ['records', 'samples', 'results', 'data']:
        if isinstance(data, dict) and isinstance(data.get(key), list):
            return data[key]
    raise ValueError(f'Unsupported result structure: {path}')


def list_candidate_result_paths():
    import glob
    candidates = []
    for pattern in CANDIDATE_RESULT_GLOBS:
        candidates.extend(glob.glob(pattern, recursive=True))
    candidates = sorted({Path(p) for p in candidates if Path(p).is_file()}, key=lambda p: str(p))
    if not candidates:
        print('No probability cache candidates found under the configured run path.')
        print('Run the eval/infer notebook until extract_probability_cache writes *_task_probability_cache.json,')
        print('or set RESULT_PATH to another JSON/JSONL that already has first_probs, last_probs, pair_probs, and gold_order.')
        return []
    rows = []
    for path in candidates:
        rows.append({
            'path': str(path),
            'size_mb': path.stat().st_size / (1024 * 1024),
            'modified': pd.to_datetime(path.stat().st_mtime, unit='s'),
        })
    display(pd.DataFrame(rows).sort_values('path'))
    return candidates


if RESULT_PATH is None:
    list_candidate_result_paths()
    raise ValueError('Set RESULT_PATH manually to the exact probability cache JSON/JSONL, then rerun from this cell.')

RESULT_PATH = Path(RESULT_PATH)
if not RESULT_PATH.exists():
    list_candidate_result_paths()
    raise FileNotFoundError(f'RESULT_PATH does not exist: {RESULT_PATH}')

samples_all = load_json_or_jsonl(RESULT_PATH)
samples = samples_all[:SAMPLE_LIMIT] if SAMPLE_LIMIT is not None else samples_all

required = {'gold_order', 'first_probs', 'last_probs', 'pair_probs'}
missing = required - set(samples[0].keys())
if missing:
    raise ValueError(f'RESULT_PATH is missing required fields in the first sample: {sorted(missing)}')

print('result_path:', RESULT_PATH)
print('loaded:', len(samples_all), 'using:', len(samples))
print('first sample keys:', sorted(samples[0].keys()))


In [ ]:
# 5) Normalization helpers
def as_int_order(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            value = json.loads(text)
        except json.JSONDecodeError:
            import ast
            value = ast.literal_eval(text)
    order = [int(x) for x in value]
    return order if len(order) == 4 and sorted(order) == FRAME_IDS else None


def normalize_prob_dict(prob_dict):
    if prob_dict is None:
        return {}
    return {str(k): float(v) for k, v in prob_dict.items()}


def endpoint_prob(prob_dict, frame):
    probs = normalize_prob_dict(prob_dict)
    for key in [str(frame), int(frame)]:
        if str(key) in probs:
            return float(probs[str(key)])
    return 0.0


def pair_prob(sample, a, b):
    pair_probs = sample.get('pair_probs') or sample.get('pairwise_probs') or {}
    keys = [
        f'{a}>{b}', f'{a}_gt_{b}', f'{a}_before_{b}', f'{a},{b}', f'{a}_{b}',
        str((a, b)), str([a, b]),
    ]
    for key in keys:
        if key in pair_probs:
            return float(pair_probs[key])
    reverse_keys = [
        f'{b}>{a}', f'{b}_gt_{a}', f'{b}_before_{a}', f'{b},{a}', f'{b}_{a}',
        str((b, a)), str([b, a]),
    ]
    for key in reverse_keys:
        if key in pair_probs:
            return 1.0 - float(pair_probs[key])
    if str(a) in pair_probs and isinstance(pair_probs[str(a)], dict) and str(b) in pair_probs[str(a)]:
        return float(pair_probs[str(a)][str(b)])
    if str(b) in pair_probs and isinstance(pair_probs[str(b)], dict) and str(a) in pair_probs[str(b)]:
        return 1.0 - float(pair_probs[str(b)][str(a)])
    raise KeyError(f'Missing pair probability P({a}>{b}) for sample {sample.get("sample_id")}')


def normalized(scores):
    scores = {int(k): max(float(v), 0.0) for k, v in scores.items()}
    total = sum(scores.values())
    if total <= 0:
        return {k: 1.0 / len(scores) for k in scores}
    return {k: v / total for k, v in scores.items()}


In [ ]:
# 6) Decoders
def pair_endpoint_probs(sample):
    pair_first_scores = {}
    pair_last_scores = {}
    for i in FRAME_IDS:
        pair_first_scores[i] = sum(pair_prob(sample, i, j) for j in FRAME_IDS if j != i)
        pair_last_scores[i] = sum(pair_prob(sample, j, i) for j in FRAME_IDS if j != i)
    return normalized(pair_first_scores), normalized(pair_last_scores)


def fuse_endpoint_probs(sample, endpoint_weight, pair_weight):
    pair_first, pair_last = pair_endpoint_probs(sample)
    fused_first = {}
    fused_last = {}
    for i in FRAME_IDS:
        fused_first[i] = endpoint_weight * endpoint_prob(sample.get('first_probs'), i) + pair_weight * pair_first[i]
        fused_last[i] = endpoint_weight * endpoint_prob(sample.get('last_probs'), i) + pair_weight * pair_last[i]
    return normalized(fused_first), normalized(fused_last), pair_first, pair_last


def select_joint_endpoints(fused_first, fused_last):
    best = None
    best_score = -float('inf')
    for first, last in itertools.permutations(FRAME_IDS, 2):
        score = math.log(fused_first[first] + EPS) + math.log(fused_last[last] + EPS)
        if score > best_score:
            best = (first, last)
            best_score = score
    return best


def middle_score(sample, middle_order):
    return sum(
        math.log(pair_prob(sample, middle_order[i], middle_order[j]) + EPS)
        for i in range(len(middle_order))
        for j in range(i + 1, len(middle_order))
    )


def decode_endpoint_pairwise(sample, endpoint_weight, pair_weight):
    fused_first, fused_last, pair_first, pair_last = fuse_endpoint_probs(sample, endpoint_weight, pair_weight)
    first, last = select_joint_endpoints(fused_first, fused_last)
    middle = [x for x in FRAME_IDS if x not in {first, last}]
    best_middle = max(itertools.permutations(middle), key=lambda order: middle_score(sample, order))
    order = [first, *best_middle, last]
    extras = {
        'pair_first_probs': pair_first,
        'pair_last_probs': pair_last,
        'fused_first': fused_first,
        'fused_last': fused_last,
        'selected_first': first,
        'selected_last': last,
    }
    return order, extras


def weighted_score(sample, order, alpha=1.0, beta=1.0, gamma=1.0):
    pair_score = np.mean([
        math.log(pair_prob(sample, order[i], order[j]) + EPS)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(endpoint_prob(sample.get('first_probs'), order[0]) + EPS)
    last_score = math.log(endpoint_prob(sample.get('last_probs'), order[-1]) + EPS)
    return alpha * pair_score + beta * first_score + gamma * last_score


def decode_current_weighted(sample):
    explicit = as_int_order(sample.get(CURRENT_DECODER_FIELD))
    if explicit is not None:
        return explicit, 'field'
    fallback = list(max(PERMUTATIONS, key=lambda order: weighted_score(
        sample, order, CURRENT_WEIGHT_ALPHA, CURRENT_WEIGHT_BETA, CURRENT_WEIGHT_GAMMA
    )))
    return fallback, 'weighted_fallback'


In [ ]:
# 7) Metrics
def order_ranks(order):
    return {int(image_number): position for position, image_number in enumerate(order)}


def relative_pair_accuracy(pred_order, gold_order):
    if pred_order is None:
        return 0.0
    pred_ranks = order_ranks(pred_order)
    gold_ranks = order_ranks(gold_order)
    return float(np.mean([
        (pred_ranks[a] < pred_ranks[b]) == (gold_ranks[a] < gold_ranks[b])
        for a, b in itertools.combinations(FRAME_IDS, 2)
    ]))


def metric_row(pred_order, gold_order):
    if pred_order is None:
        return {
            'exact': 0.0, 'first': 0.0, 'last': 0.0, 'both_endpoints': 0.0,
            'position': 0.0, 'relative_pair': 0.0, 'valid': 0.0,
        }
    return {
        'exact': float(pred_order == gold_order),
        'first': float(pred_order[0] == gold_order[0]),
        'last': float(pred_order[-1] == gold_order[-1]),
        'both_endpoints': float(pred_order[0] == gold_order[0] and pred_order[-1] == gold_order[-1]),
        'position': float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        'relative_pair': relative_pair_accuracy(pred_order, gold_order),
        'valid': 1.0,
    }


def summarize(prediction_df):
    metric_cols = ['exact', 'first', 'last', 'both_endpoints', 'position', 'relative_pair', 'valid']
    return prediction_df.groupby('decoder', as_index=False)[metric_cols].mean()


In [ ]:
# 8) Run 100-sample comparison and save core outputs
sample_rows = []
prediction_rows = []
current_sources = []

for sample in samples:
    sample_id = str(sample.get('sample_id', sample.get('Id', sample.get('id', len(sample_rows)))))
    gold_order = as_int_order(sample.get('gold_order') or sample.get('order'))
    if gold_order is None:
        raise ValueError(f'Missing/invalid gold_order for sample {sample_id}')

    current_order, current_source = decode_current_weighted(sample)
    current_sources.append(current_source)
    ep50_order, ep50 = decode_endpoint_pairwise(sample, 0.5, 0.5)
    ep40_order, ep40 = decode_endpoint_pairwise(sample, 0.4, 0.6)
    direct_order = as_int_order(sample.get('direct_order'))

    decoder_orders = {
        'current_weighted_permutation': current_order,
        'endpoint_pair_50_50': ep50_order,
        'endpoint_pair_40_60': ep40_order,
    }
    if direct_order is not None:
        decoder_orders['direct_order'] = direct_order

    for decoder, pred_order in decoder_orders.items():
        row = metric_row(pred_order, gold_order)
        row.update({'sample_id': sample_id, 'decoder': decoder, 'pred_order': pred_order, 'gold_order': gold_order})
        prediction_rows.append(row)

    sample_rows.append({
        'sample_id': sample_id,
        'gold_order': gold_order,
        'current_order': current_order,
        'endpoint_pair_50_50_order': ep50_order,
        'endpoint_pair_40_60_order': ep40_order,
        'direct_order': direct_order,
        'pair_first_probs': {str(k): v for k, v in ep40['pair_first_probs'].items()},
        'pair_last_probs': {str(k): v for k, v in ep40['pair_last_probs'].items()},
        'fused_first_50_50': {str(k): v for k, v in ep50['fused_first'].items()},
        'fused_last_50_50': {str(k): v for k, v in ep50['fused_last'].items()},
        'fused_first_40_60': {str(k): v for k, v in ep40['fused_first'].items()},
        'fused_last_40_60': {str(k): v for k, v in ep40['fused_last'].items()},
        'selected_first_50_50': ep50['selected_first'],
        'selected_last_50_50': ep50['selected_last'],
        'selected_first_40_60': ep40['selected_first'],
        'selected_last_40_60': ep40['selected_last'],
        'current_source': current_source,
    })

prediction_df = pd.DataFrame(prediction_rows)
summary_df = summarize(prediction_df)

base = summary_df[summary_df['decoder'].eq('current_weighted_permutation')].iloc[0]
for col in ['exact', 'first', 'last', 'both_endpoints', 'position', 'relative_pair']:
    summary_df[f'{col}_delta'] = summary_df[col] - float(base[col])

summary_path = OUTPUT_DIR / 'decoding_comparison_summary.csv'
sample_json_path = OUTPUT_DIR / 'decoding_sample_results.json'
prediction_path = OUTPUT_DIR / 'decoding_predictions_long.csv'
summary_df.to_csv(summary_path, index=False)
prediction_df.to_csv(prediction_path, index=False)
with sample_json_path.open('w', encoding='utf-8') as f:
    json.dump(sample_rows, f, ensure_ascii=False, indent=2)

print('current decoder sources:', pd.Series(current_sources).value_counts().to_dict())
print('saved:', summary_path)
print('saved:', prediction_path)
print('saved:', sample_json_path)
display(summary_df.sort_values(['exact', 'both_endpoints', 'relative_pair', 'position'], ascending=False))


In [ ]:
# 9) Improvement / degradation analysis
def exact_map(decoder):
    return prediction_df[prediction_df['decoder'].eq(decoder)].set_index('sample_id')['exact']

base_exact = exact_map('current_weighted_permutation')
analysis_rows = []
improved_all = []
degraded_all = []
for decoder in ['endpoint_pair_50_50', 'endpoint_pair_40_60']:
    cand = exact_map(decoder)
    improved_ids = cand[(base_exact == 0.0) & (cand == 1.0)].index.tolist()
    degraded_ids = cand[(base_exact == 1.0) & (cand == 0.0)].index.tolist()

    improved = pd.DataFrame([row | {'decoder': decoder} for row in sample_rows if row['sample_id'] in improved_ids])
    degraded = pd.DataFrame([row | {'decoder': decoder} for row in sample_rows if row['sample_id'] in degraded_ids])

    improved.to_csv(OUTPUT_DIR / f'improved_samples_{decoder}.csv', index=False)
    degraded.to_csv(OUTPUT_DIR / f'degraded_samples_{decoder}.csv', index=False)
    improved_all.append(improved)
    degraded_all.append(degraded)

    analysis_rows.append({'decoder': decoder, 'improved_exact_count': len(improved_ids), 'degraded_exact_count': len(degraded_ids)})

if improved_all:
    pd.concat(improved_all, ignore_index=True).to_csv(OUTPUT_DIR / 'improved_samples.csv', index=False)
if degraded_all:
    pd.concat(degraded_all, ignore_index=True).to_csv(OUTPUT_DIR / 'degraded_samples.csv', index=False)

analysis_df = pd.DataFrame(analysis_rows)
analysis_df.to_csv(OUTPUT_DIR / 'improvement_degradation_counts.csv', index=False)
display(analysis_df)


In [ ]:
# 10) Endpoint error type analysis
def endpoint_transition_type(base_order, new_order, gold_order):
    base_first = base_order[0] == gold_order[0]
    base_last = base_order[-1] == gold_order[-1]
    new_first = new_order[0] == gold_order[0]
    new_last = new_order[-1] == gold_order[-1]
    if (not base_first and new_first) and (not base_last and new_last):
        return 'first_and_last_improved'
    if (not base_first and new_first) and base_last == new_last:
        return 'first_only_improved'
    if base_first == new_first and (not base_last and new_last):
        return 'last_only_improved'
    if (not base_first and new_first) and (base_last and not new_last):
        return 'first_improved_last_degraded'
    if (base_first and not new_first) and (not base_last and new_last):
        return 'last_improved_first_degraded'
    if base_order[0] == new_order[0] and base_order[-1] == new_order[-1]:
        if base_order != new_order:
            return 'same_endpoints_middle_changed'
        return 'same_order'
    if base_first == new_first and base_last == new_last:
        return 'endpoints_changed_same_correctness'
    if (base_first and not new_first) or (base_last and not new_last):
        return 'endpoint_degraded'
    return 'other'

endpoint_rows = []
for row in sample_rows:
    gold = row['gold_order']
    base_order = row['current_order']
    for decoder, key in [('endpoint_pair_50_50', 'endpoint_pair_50_50_order'), ('endpoint_pair_40_60', 'endpoint_pair_40_60_order')]:
        new_order = row[key]
        endpoint_rows.append({
            'sample_id': row['sample_id'],
            'decoder': decoder,
            'error_type': endpoint_transition_type(base_order, new_order, gold),
            'base_exact': float(base_order == gold),
            'new_exact': float(new_order == gold),
            'base_order': base_order,
            'new_order': new_order,
            'gold_order': gold,
        })

endpoint_error_df = pd.DataFrame(endpoint_rows)
endpoint_error_df.to_csv(OUTPUT_DIR / 'endpoint_error_analysis.csv', index=False)
display(endpoint_error_df.groupby(['decoder', 'error_type']).size().reset_index(name='count').sort_values(['decoder', 'count'], ascending=[True, False]))
